In [6]:
import numpy as np
from tqdm import tqdm
import glob
import re
import json
from typing import Dict, List, Optional, Tuple, Union
import os
import inspect
from copy import deepcopy

import pandas as pd

In [7]:
def get_gt(file_path:str, word_num:int, 
           rename = {'Image.no':'image_no','print size':'print_size'},
           L_R:bool = False, human=False) -> pd.DataFrame:
    # Specify the path to your Excel file
    
    # Read the Excel file into a pandas DataFrame
    xls = pd.ExcelFile(file_path)
    # 获取所有sheet的名称
    sheet_names = xls.sheet_names

    # 逐个读取sheet，并存入列表
    dfs = [pd.read_excel(xls, sheet_name=sheet) for sheet in sheet_names]

    # 合并所有的DataFrame
    gt = pd.concat(dfs, ignore_index=True)
    
    # df = pd.read_excel(file_path, sheet_name=None, header=0)
    
    cols = ['SubID','print_size','image_no','a','b'] + ['L{}'.format(num) for num in range(1,word_num+1)]
    # if human:
    #     cols = ['SubID'] + cols
    
    if L_R:
        cols += ['R{}'.format(num) for num in range(1,word_num+1)]

    # gt = pd.concat(df,axis=0,ignore_index=True)
    if rename is not None:
        gt.rename(columns={'Image.no':'image_no','print size':'print_size'}, inplace=True)
    gt['print_size'] = np.round(gt['print_size'],decimals=1)
    gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
    
    cols = ['row'] + cols
    gt = gt.loc[:,cols].set_index(['print_size','image_no'])

    
    if human:
        return gt


    # 找到重复的索引
    duplicate_index = gt.index.duplicated()

    # 过滤出不重复的行
    df_no_duplicates = gt[~duplicate_index]
    return df_no_duplicates

In [80]:
import pandas as pd
import json

def dataframe_to_coco(df):
    """
    Convert a DataFrame to COCO captioning format.
    
    Assumptions:
      - The image path is constructed by concatenating the values from the "a" and "row" columns.
      - The caption is built by concatenating the text from columns L1 to L15 (ignoring NaNs).
    """
    coco = {
        "images": [],
        "annotations": [],
        "info": {
            "description": "MNRead Dataset",
            "version": "1.0"
        },
        "licenses": []
    }
    
    image_id = 0
    annotation_id = 0
    word_num = max([int(i[1:]) for i in df.columns if 'L' in i])

    # Iterate through each row in the dataframe
    for _, row in df.iterrows():
        # Construct the image file name. Here we assume column 'a' is the image number
        # and 'row' is another identifier. Adjust formatting as needed.
        file_name = f"{row['image_no']}_512_{int(row['row'])}.jpg"
        
        # Build caption by concatenating L1 to L15 (ignoring NaN values)
        caption_parts = []
        for i in range(1, word_num+1):
            col_name = f"L{i}"
            token = row.get(col_name)
            if pd.notnull(token):
                caption_parts.append(str(token))
        caption = " ".join(caption_parts)
        
        # Add image entry (you may add width, height if known)
        coco["images"].append({
            "id": image_id,
            "file_name": file_name,
            "Filter_no": row['Filter_no']
        })
        
        # Add annotation entry for the caption
        coco["annotations"].append({
            "id": annotation_id,
            "image_id": image_id,
            "caption": caption
        })
        
        image_id += 1
        annotation_id += 1
        
    return coco



In [77]:
filter_df = pd.read_csv('../../data/human/SelectedFilter.csv')

xlsx_file_path = '../../data/human/MNREADChartScoreSheet.xlsx'
WORD_NUM = 15
 
df_no_duplicates = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM)
df_no_duplicates.drop(columns=['SubID'], inplace=True)
df_no_duplicates['a'] = df_no_duplicates['a'].round(3)
df_no_duplicates['b'] = df_no_duplicates['b'].round(3)
df_no_duplicates['a'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['b'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['a'].replace(0.287, 0.288, inplace=True)
df_no_duplicates['b'].replace(0.287, 0.288, inplace=True)
df_no_duplicates.reset_index(inplace=True)
df_no_duplicates = pd.merge(left=df_no_duplicates,right=filter_df[['a','b','Filter_no']],how='inner',on=['a','b'])

/var/folders/_n/m3nnfh793dz6s8wg4lfcfvyr0000gn/T/ipykernel_53608/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/var/folders/_n/m3nnfh793dz6s8wg4lfcfvyr0000gn/T/ipykernel_53608/3183782216.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[co

In [78]:
coco_dict = dataframe_to_coco(df_no_duplicates)
    
# Save the COCO JSON to a file
with open("../../data/mnread/anno.json", "w") as f:
    json.dump(coco_dict, f, indent=4)
    

In [81]:
xlsx_file_path = '../../data/human/LetterChartScoreSheet.xlsx'
WORD_NUM = 5
 
df_no_duplicates = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM)
df_no_duplicates.drop(columns=['SubID'], inplace=True)
df_no_duplicates['a'] = df_no_duplicates['a'].round(3)
df_no_duplicates['b'] = df_no_duplicates['b'].round(3)
df_no_duplicates['a'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['b'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['a'].replace(0.287, 0.288, inplace=True)
df_no_duplicates['b'].replace(0.287, 0.288, inplace=True)
df_no_duplicates.reset_index(inplace=True)
df_no_duplicates = pd.merge(left=df_no_duplicates,right=filter_df[['a','b','Filter_no']],how='inner',on=['a','b'])
coco_dict = dataframe_to_coco(df_no_duplicates)
    
# Save the COCO JSON to a file
with open("../../data/etdrs/anno.json", "w") as f:
    json.dump(coco_dict, f, indent=4)

/var/folders/_n/m3nnfh793dz6s8wg4lfcfvyr0000gn/T/ipykernel_53608/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/var/folders/_n/m3nnfh793dz6s8wg4lfcfvyr0000gn/T/ipykernel_53608/4045801704.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

In [82]:
df_no_duplicates

,print_size,image_no,row,a,b,L1,L2,L3,L4,L5,Filter_no
0,1.7,Chart1,1.0,1.0,1.000,C,V,D,O,S,16
1,1.6,Chart1,2.0,1.0,1.000,C,O,H,K,N,16
2,1.5,Chart1,3.0,1.0,1.000,V,R,H,Z,C,16
3,1.4,Chart1,4.0,1.0,1.000,O,R,H,Z,V,16
4,1.3,Chart1,5.0,1.0,1.000,N,K,O,Z,R,16
...,...,...,...,...,...,...,...,...,...,...,...
493,0.1,Chart54,17.0,1.0,0.011,N,H,S,V,Z,11
494,0.0,Chart54,18.0,1.0,0.011,V,H,D,R,Z,11
495,-0.1,Chart54,19.0,1.0,0.011,R,N,V,H,K,11
496,-0.2,Chart54,20.0,1.0,0.011,Z,C,K,R,V,11
